In [0]:
%sql
 USE CATALOG day8_catalog;
USE SCHEMA ecommerce;


In [0]:
%sql
EXPLAIN EXTENDED
SELECT *
FROM events_silver
WHERE event_type = 'purchase';


In [0]:
%sql
CREATE OR REPLACE TABLE events_silver_part
USING DELTA
PARTITIONED BY (event_date, event_type)
AS
SELECT
  *,
  DATE(event_time) AS event_date
FROM events_silver;


In [0]:
%sql
DESCRIBE DETAIL events_silver_part;


In [0]:
%sql
EXPLAIN EXTENDED
SELECT *
FROM events_silver_part
WHERE event_type = 'purchase'
  AND event_date >= '2019-11-05';


In [0]:
%sql
OPTIMIZE events_silver_part
ZORDER BY (user_id, product_id);


In [0]:
import time

# Before optimization (original table)
start = time.time()
spark.sql("""
SELECT *
FROM day8_catalog.ecommerce.events_silver
WHERE user_id = 12345
""").count()
print(f"Before optimization: {time.time() - start:.2f} seconds")


In [0]:
# After optimization (partitioned + ZORDER)
start = time.time()
spark.sql("""
SELECT *
FROM day8_catalog.ecommerce.events_silver_part
WHERE user_id = 12345
""").count()
print(f"After optimization: {time.time() - start:.2f} seconds")


In [0]:
%sql
OPTIMIZE events_silver_part
ZORDER BY (user_id, product_id);


In [0]:
%sql
SELECT *
FROM events_silver_part
WHERE user_id = 12345;
